# 2. Tokenize transcriptomes and data pairing



This notebook tokenizes and pairs the pre-built LPS atlas. Run it separately for two workflows and keep their output directories distinct:
- `resting_all_times`: `normal` source; 90m/6h/10h targets; used by notebooks 03–05 and JEPA.
- `il1b_ko_90m`: `90m_LPS` source; 6h/10h targets; used by notebooks 06–07 after separate model training.

Target numbers are 1-based slots created by this tokenization, not universal time labels. Always verify the generated target filenames.

## 2.1. Configure paths and parameters

These parameters mirror the CLI options in `perturbgen.pp.GF_tokenisation`.


In order to download data, including the LPS data for this tutorial see https://perturbgen.cog.sanger.ac.uk/docs/data.html

In [9]:
# download LPS data from AWS S3 bucket
!aws --endpoint-url https://cog.sanger.ac.uk --no-sign-request \
  s3 cp s3://perturbgen/Manuscript/lps_otar.h5ad ./lps_otar.h5ad

download: s3://perturbgen/Manuscript/lps_otar.h5ad to ./lps_otar.h5ad


In [4]:
from pathlib import Path

REPO_ROOT = Path("/home/stuke1/perturbgen/Perturbgen")

H5AD_PATH = str(REPO_ROOT / "docs/examples/lps_otar.h5ad")

# Select one workflow; never mix their tokenized outputs or target numbering.
WORKFLOW = "il1b_ko_90m"  #"resting_all_times"  # or "il1b_ko_90m"
WORKFLOW_PRESETS = {
    "resting_all_times": {
        "dataset_name": "LPS_all_tps_2k",
        "reference_time": "normal",
        "time_point_order": ["normal", "90m_LPS", "6h_LPS", "10h_LPS"],
    },
    "il1b_ko_90m": {
        "dataset_name": "lps_90min_perturb",
        "reference_time": "90m_LPS",
        "time_point_order": ["90m_LPS", "6h_LPS", "10h_LPS"],
    },
}
if WORKFLOW not in WORKFLOW_PRESETS:
    raise ValueError(f"Unknown WORKFLOW: {WORKFLOW}")
preset = WORKFLOW_PRESETS[WORKFLOW]
DATASET_NAME = preset["dataset_name"]
REFERENCE_TIME = preset["reference_time"]
TIME_POINT_ORDER = preset["time_point_order"]
print("workflow:", WORKFLOW, "| source:", REFERENCE_TIME, "| order:", TIME_POINT_ORDER)

GENE_FILTERING_MODE = "hvg"  # one of: hvg, degs, all
HVG_MODE = "before_tokenisation"  # before_tokenisation or after_tokenisation
VAR_LIST = ["cell_type_harmonized", "time_after_LPS"]
PAIRING_MODE = "stratified"
TIME_OBS = "time_after_LPS"
PAIRING_FILE = "path/to/pairing.csv"  # only for mapping mode
MAIN_PAIRING_OBS = "cell_type_harmonized"
OPT_PAIRING_OBS = []
NPROC = 8
N_HVG = 2000

# pretrained Geneformer 95M dicts shipped with the repo
GENE_MEDIAN_PATH = str(REPO_ROOT / "perturbgen/pp/gene_median_dict_gftokens_gc95M.pkl")
TOKEN_DICT_PATH = str(REPO_ROOT / "perturbgen/pp/token_dict_gftokens_gc95M.pkl")
GENE_MAPPING_PATH = str(REPO_ROOT / "perturbgen/pp/ensembl_mapping_dict_gc95M.pkl")


workflow: il1b_ko_90m | source: 90m_LPS | order: ['90m_LPS', '6h_LPS', '10h_LPS']


### Workflow-specific source and target numbering

`REFERENCE_TIME` becomes the source. Every other entry in that preset's `TIME_POINT_ORDER` becomes a 1-based target slot.

- `resting_all_times`: source `normal`; targets `1_90m_LPS`, `2_6h_LPS`, `3_10h_LPS`; use `pred_tps=[1,2,3]`.
- `il1b_ko_90m`: source `90m_LPS`; targets `1_6h_LPS`, `2_10h_LPS`; use `pred_tps=[1,2]`.

A paper-faithful IL1B source-KO experiment trains a separate MaskGIT decoder and matching count head on the second tokenization. It is not created by loading the resting-source notebook-03 checkpoint with different YAML paths.

IL1B is approximately absent at `normal`, so a source knockout needs the induced 90m state. Saved outputs in this notebook may reflect an older preset until cells are rerun; current configuration and generated filenames are authoritative.

## 2.2. Build the tokenization command

This prints the exact command that will be executed. Review it before running.


In [5]:
cmd = [
    "python",
    "-m",
    "perturbgen",
    "tokenise",
    "--h5ad_path", H5AD_PATH,
    "--dataset", DATASET_NAME,
    "--gene_filtering_mode", GENE_FILTERING_MODE,
    "--hvg_mode", HVG_MODE,
    "--var_list", *VAR_LIST,
    "--pairing_mode", PAIRING_MODE,
    "--time_obs", TIME_OBS,
    "--main_pairing_obs", MAIN_PAIRING_OBS,
    "--nproc", str(NPROC),
    "--n_hvg", str(N_HVG),
    "--reference_time", REFERENCE_TIME,
    "--time_point_order", *TIME_POINT_ORDER,
    "--gene_median_path", GENE_MEDIAN_PATH,
    "--token_dict_path", TOKEN_DICT_PATH,
    "--gene_mapping_path", GENE_MAPPING_PATH,
]

if PAIRING_MODE == "mapping":
    cmd += ["--pairing_file", PAIRING_FILE]
if OPT_PAIRING_OBS:
    cmd += ["--opt_pairing_obs", *OPT_PAIRING_OBS]

print(" ".join(cmd))


python -m perturbgen tokenise --h5ad_path /home/stuke1/perturbgen/Perturbgen/docs/examples/lps_otar.h5ad --dataset lps_90min_perturb --gene_filtering_mode hvg --hvg_mode before_tokenisation --var_list cell_type_harmonized time_after_LPS --pairing_mode stratified --time_obs time_after_LPS --main_pairing_obs cell_type_harmonized --nproc 8 --n_hvg 2000 --reference_time 90m_LPS --time_point_order 90m_LPS 6h_LPS 10h_LPS --gene_median_path /home/stuke1/perturbgen/Perturbgen/perturbgen/pp/gene_median_dict_gftokens_gc95M.pkl --token_dict_path /home/stuke1/perturbgen/Perturbgen/perturbgen/pp/token_dict_gftokens_gc95M.pkl --gene_mapping_path /home/stuke1/perturbgen/Perturbgen/perturbgen/pp/ensembl_mapping_dict_gc95M.pkl


## 2.3. Run tokenization (CPU-only)

This step can take time depending on dataset size.


In [6]:
import subprocess

subprocess.run(cmd, check=True)


loading, please wait...
Current working directory: /home/stuke1/perturbgen
Start preprocessing adata...
Number of genes dropped: 0
Finished preprocessing adata.
Start tokenisation of adata...
Tokenizing /mnt/sod2-project/csb4/stuke1/perturbgen_reproduction/lps/tokenized_data/lps_90min_perturb/h5ad_pairing_2000_hvg_GF_genes/lps_90min_perturb.h5ad


/home/stuke1/perturbgen/Perturbgen/perturbgen/pp/tokenizer.py:495: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  for i in adata.var["ensembl_id_collapsed"][coding_miRNA_loc]
/home/stuke1/perturbgen/Perturbgen/perturbgen/pp/tokenizer.py:498: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  coding_miRNA_ids = adata.var["ensembl_id_collapsed"][coding_miRNA_loc]


/mnt/sod2-project/csb4/stuke1/perturbgen_reproduction/lps/tokenized_data/lps_90min_perturb/h5ad_pairing_2000_hvg_GF_genes/lps_90min_perturb.h5ad has no column attribute 'filter_pass'; tokenizing all cells.
Creating dataset.


Saving the dataset (1/1 shards): 100%|██████████| 223478/223478 [00:11<00:00, 18992.40 examples/s]  


Finished tokenisation.


/home/stuke1/perturbgen/Perturbgen/perturbgen/src/utils.py:1571: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  adata_obs_.groupby(grouping_obs)[time_obs].transform('nunique') == total_tps
/home/stuke1/perturbgen/Perturbgen/perturbgen/src/utils.py:1574: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  grouped = adata_grouped.groupby(grouping_obs)
  0%|          | 0/3 [00:00<?, ?it/s]/home/stuke1/perturbgen/.venv/lib/python3.11/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/home/stuke1

CompletedProcess(args=['python', '-m', 'perturbgen', 'tokenise', '--h5ad_path', '/home/stuke1/perturbgen/Perturbgen/docs/examples/lps_otar.h5ad', '--dataset', 'lps_90min_perturb', '--gene_filtering_mode', 'hvg', '--hvg_mode', 'before_tokenisation', '--var_list', 'cell_type_harmonized', 'time_after_LPS', '--pairing_mode', 'stratified', '--time_obs', 'time_after_LPS', '--main_pairing_obs', 'cell_type_harmonized', '--nproc', '8', '--n_hvg', '2000', '--reference_time', '90m_LPS', '--time_point_order', '90m_LPS', '6h_LPS', '10h_LPS', '--gene_median_path', '/home/stuke1/perturbgen/Perturbgen/perturbgen/pp/gene_median_dict_gftokens_gc95M.pkl', '--token_dict_path', '/home/stuke1/perturbgen/Perturbgen/perturbgen/pp/token_dict_gftokens_gc95M.pkl', '--gene_mapping_path', '/home/stuke1/perturbgen/Perturbgen/perturbgen/pp/ensembl_mapping_dict_gc95M.pkl'], returncode=0)

## 2.4. Outputs

Tokenized files are written under the tokenized data directory defined in `perturbgen/configs/paths.py`.
You should see a new folder for your dataset name containing `.dataset` files and pairing outputs in `perturbgen_reproduction/lps/tokenized_data`.

After §2.6 you will also get a frozen `splits/` artifact (90% train / 10% val indices) for all downstream train / metrics / embedding runs.


---
After inspection, run **§2.6** to freeze a reproducible **90% train / 10% val** split (no held-out test) next to the tokenized data.
Then continue to Notebook 3 for training.


## 2.5. Dataset inspection (size, statistics, example pairings)

Run this after tokenization to quickly inspect:
- dataset sizes (`src` + each `tgt` timepoint)
- sequence-length statistics
- pairing integrity (overlap/missing/duplicates)
- a few concrete source-target pairing examples

In [3]:
from __future__ import annotations

from pathlib import Path
import random

import numpy as np
import pandas as pd
from datasets import load_from_disk
from IPython.display import display

# Reuse REPO_ROOT and DATASET_NAME from section 2.1
# T_perturb moved off home to sod2.
LPS_DATA = Path("/mnt/sod2-project/csb4/stuke1/perturbgen_reproduction/lps")
TOKENIZED_ROOT = LPS_DATA / "tokenized_data" / DATASET_NAME
SRC_PATH = TOKENIZED_ROOT / "dataset_2000_hvg_src" / f"{REFERENCE_TIME}.dataset"
TGT_DIR = TOKENIZED_ROOT / "dataset_2000_hvg_tgt"

print("TOKENIZED_ROOT:", TOKENIZED_ROOT)
print("SRC_PATH exists:", SRC_PATH.exists())
print("TGT_DIR exists:", TGT_DIR.exists())

if not SRC_PATH.exists() or not TGT_DIR.exists():
    raise FileNotFoundError(
        "Tokenized outputs not found. Run section 2.3 first or verify DATASET_NAME."
    )

src_ds = load_from_disk(str(SRC_PATH))
tgt_paths = sorted(TGT_DIR.glob("*.dataset"))
if not tgt_paths:
    raise FileNotFoundError(f"No *.dataset found under {TGT_DIR}")

tgt_ds = {p.stem: load_from_disk(str(p)) for p in tgt_paths}


def _length_stats(input_ids_col):
    lens = np.asarray([len(x) for x in input_ids_col], dtype=np.int32)
    return {
        "len_min": int(lens.min()),
        "len_mean": float(lens.mean()),
        "len_median": float(np.median(lens)),
        "len_max": int(lens.max()),
    }


def _pairing_stats(ds, src_pair_set):
    if "cell_pairing_index" not in ds.column_names:
        return {
            "has_pairing_index": False,
            "n_unique_pair": np.nan,
            "n_dup_rows": np.nan,
            "n_missing_in_src": np.nan,
            "pct_missing_in_src": np.nan,
        }

    pair_idx = np.asarray(ds["cell_pairing_index"])
    unique_pair = set(pair_idx.tolist())
    missing = unique_pair.difference(src_pair_set)

    return {
        "has_pairing_index": True,
        "n_unique_pair": int(len(unique_pair)),
        "n_dup_rows": int(len(pair_idx) - len(unique_pair)),
        "n_missing_in_src": int(len(missing)),
        "pct_missing_in_src": round(100.0 * len(missing) / max(len(unique_pair), 1), 4),
    }


# -------- Summary table --------
summary_rows = []

src_pair = np.asarray(src_ds["cell_pairing_index"]) if "cell_pairing_index" in src_ds.column_names else np.array([])
src_pair_set = set(src_pair.tolist())

src_stats = _length_stats(src_ds["input_ids"])
summary_rows.append(
    {
        "split": "src",
        "name": SRC_PATH.stem,
        "n_rows": int(len(src_ds)),
        "n_columns": int(len(src_ds.column_names)),
        **src_stats,
        **_pairing_stats(src_ds, src_pair_set),
    }
)

for name, ds in tgt_ds.items():
    lstats = _length_stats(ds["input_ids"])
    summary_rows.append(
        {
            "split": "tgt",
            "name": name,
            "n_rows": int(len(ds)),
            "n_columns": int(len(ds.column_names)),
            **lstats,
            **_pairing_stats(ds, src_pair_set),
        }
    )

summary_df = pd.DataFrame(summary_rows).sort_values(["split", "name"]).reset_index(drop=True)
print("\n=== Dataset size + statistics ===")
display(summary_df)

print("\nColumns in src:")
print(src_ds.column_names)
print("\nColumns in each tgt:")
for name, ds in tgt_ds.items():
    print(f"- {name}: {ds.column_names}")


# -------- Example pairings --------
print("\n=== Example source-target pairings ===")
if "cell_pairing_index" not in src_ds.column_names:
    print("No cell_pairing_index in src, cannot show pair examples.")
else:
    src_lookup = {}
    for i, pair_id in enumerate(src_ds["cell_pairing_index"]):
        # keep first occurrence if duplicates exist
        src_lookup.setdefault(pair_id, i)

    rng = random.Random(42)
    example_rows = []
    max_per_tgt = 4

    for tgt_name, ds in tgt_ds.items():
        if "cell_pairing_index" not in ds.column_names:
            continue

        candidate_ids = [pid for pid in ds["cell_pairing_index"] if pid in src_lookup]
        if not candidate_ids:
            continue
        sample_ids = rng.sample(candidate_ids, min(max_per_tgt, len(candidate_ids)))

        for pair_id in sample_ids:
            srow = src_ds[src_lookup[pair_id]]
            # choose first matching row from this tgt split
            tidx = next(i for i, pid in enumerate(ds["cell_pairing_index"]) if pid == pair_id)
            trow = ds[tidx]

            example_rows.append(
                {
                    "tgt_split": tgt_name,
                    "pair_id": int(pair_id),
                    "src_len": int(len(srow["input_ids"])),
                    "tgt_len": int(len(trow["input_ids"])),
                    "src_cell_type": srow.get("cell_type_harmonized", None),
                    "tgt_cell_type": trow.get("cell_type_harmonized", None),
                    "src_time": srow.get("time_after_LPS", None),
                    "tgt_time": trow.get("time_after_LPS", None),
                }
            )

    if example_rows:
        display(pd.DataFrame(example_rows))
    else:
        print("No examples could be built from current datasets.")


# -------- Optional: quick class balance snapshot --------
print("\n=== Quick class-balance snapshot (cell_type_harmonized, top 10 each tgt) ===")
for tgt_name, ds in tgt_ds.items():
    if "cell_type_harmonized" not in ds.column_names:
        continue
    vc = pd.Series(ds["cell_type_harmonized"]).value_counts().head(10)
    print(f"\n[{tgt_name}] top-10 cell types")
    display(vc.to_frame("count"))


TOKENIZED_ROOT: /mnt/sod2-project/csb4/stuke1/perturbgen_reproduction/lps/tokenized_data/LPS_all_tps_2k
SRC_PATH exists: True
TGT_DIR exists: True

=== Dataset size + statistics ===


,split,name,n_rows,n_columns,len_min,len_mean,len_median,len_max,has_pairing_index,n_unique_pair,n_dup_rows,n_missing_in_src,pct_missing_in_src
0,src,normal,148107,5,19,161.689934,153.0,679,True,148107,0,0,0.0
1,tgt,1_90m_LPS,148107,5,27,151.779504,146.0,750,True,10633,137474,10633,100.0
2,tgt,2_6h_LPS,148107,5,30,150.796282,137.0,668,True,39188,108919,39188,100.0
3,tgt,3_10h_LPS,148107,5,26,140.041841,133.0,514,True,19713,128394,19713,100.0



Columns in src:
['input_ids', 'cell_type_harmonized', 'time_after_LPS', 'cell_pairing_index', 'length']

Columns in each tgt:
- 1_90m_LPS: ['input_ids', 'cell_type_harmonized', 'time_after_LPS', 'cell_pairing_index', 'length']
- 2_6h_LPS: ['input_ids', 'cell_type_harmonized', 'time_after_LPS', 'cell_pairing_index', 'length']
- 3_10h_LPS: ['input_ids', 'cell_type_harmonized', 'time_after_LPS', 'cell_pairing_index', 'length']

=== Example source-target pairings ===
No examples could be built from current datasets.

=== Quick class-balance snapshot (cell_type_harmonized, top 10 each tgt) ===

[1_90m_LPS] top-10 cell types


,count
CD4+ T cells,42257
CD8+ T cells,30231
CD14 monocytes,21363
NK,17987
B cell,10859
gamma-delta T cell,6947
mucosal invariant T cell,4947
CD16 monocytes,4085
Dendritic cells,2748
NKT,2124



[2_6h_LPS] top-10 cell types


,count
CD4+ T cells,42257
CD8+ T cells,30231
CD14 monocytes,21363
NK,17987
B cell,10859
gamma-delta T cell,6947
mucosal invariant T cell,4947
CD16 monocytes,4085
Dendritic cells,2748
NKT,2124



[3_10h_LPS] top-10 cell types


,count
CD4+ T cells,42257
CD8+ T cells,30231
CD14 monocytes,21363
NK,17987
B cell,10859
gamma-delta T cell,6947
mucosal invariant T cell,4947
CD16 monocytes,4085
Dendritic cells,2748
NKT,2124


## 2.6. Freeze train / val split (90 / 10)

Tokenization does **not** write separate train/val folders. Downstream code can recreate
indices at runtime; this cell freezes them so Notebook 3 and later analyses share one split.

Recipe matches `train.py` stratified split (`seed=42` hardcoded there — not CLI `--seed`):

| Setting | Value |
|---------|--------|
| Mode | stratified |
| Groups | `cell_type_harmonized` |
| Proportions | train **0.9** / test **0.0** / val remainder (**~0.1**) |
| Seed | **42** |
| Reference AnnData | first target timepoint h5ad under `h5ad_pairing_*_tgt` (row order = dataset row order) |

**Outputs** (under the tokenized dataset root):

```
.../LPS_all_tps_2k/splits/
  stratified_cell_type_harmonized_seed42_90_10.pkl
  stratified_cell_type_harmonized_seed42_90_10.json
  README.md
```

There is **no test split**. Validation is the 10% hold-out for early stopping / curves.


In [ ]:
from __future__ import annotations

import json
import pickle
from datetime import datetime, timezone
from pathlib import Path

import anndata as ad
import numpy as np
import pandas as pd

from perturbgen.src.utils import stratified_split

# Resolve tokenized root (same layout as §2.5).
WORKSPACE = Path("/home/stuke1/perturbgen")
LPS_DATA = Path("/mnt/sod2-project/csb4/stuke1/perturbgen_reproduction/lps")  # T_perturb moved off home
TOKENIZED_ROOT = LPS_DATA / "tokenized_data" / DATASET_NAME
TGT_H5AD_DIR = TOKENIZED_ROOT / f"h5ad_pairing_{N_HVG}_hvg_tgt"
SPLIT_DIR = TOKENIZED_ROOT / "splits"
SPLIT_DIR.mkdir(parents=True, exist_ok=True)

# Match train.py masking split (stratified + seed=42 hardcoded there).
# 90% train / 10% val / no test (test_prop=0 → remainder is val).
SPLIT_SEED = 42
TRAIN_PROP = 0.9
TEST_PROP = 0.0
SPLIT_OBS = ["cell_type_harmonized"]

tgt_h5ads = sorted(TGT_H5AD_DIR.glob("*.h5ad"))
if not tgt_h5ads:
    raise FileNotFoundError(
        f"No target h5ad under {TGT_H5AD_DIR}. Run tokenization (§2.3) first."
    )
# train/val use the first pred timepoint AnnData as the index reference (same as train.py / val.py).
ref_h5ad_path = tgt_h5ads[0]
adata = ad.read_h5ad(ref_h5ad_path)
missing = [c for c in SPLIT_OBS if c not in adata.obs.columns]
if missing:
    raise KeyError(f"Split obs missing from {ref_h5ad_path.name}: {missing}")

train_indices, val_indices, test_indices = stratified_split(
    tgt_adata=adata,
    train_prop=TRAIN_PROP,
    test_prop=TEST_PROP,
    groups=SPLIT_OBS,
    seed=SPLIT_SEED,
)
train_indices = np.asarray(train_indices, dtype=np.int64)
val_indices = np.asarray(val_indices, dtype=np.int64)
test_indices = np.asarray(test_indices, dtype=np.int64)

# Sanity: disjoint + full cover of row range is not required (rounding), but no overlap.
for a, b, name in [
    (train_indices, val_indices, "train∩val"),
    (train_indices, test_indices, "train∩test"),
    (val_indices, test_indices, "val∩test"),
]:
    overlap = np.intersect1d(a, b)
    if overlap.size:
        raise AssertionError(f"Leakage in {name}: {overlap.size} shared indices")

stem = (
    f"stratified_{'_'.join(SPLIT_OBS)}_seed{SPLIT_SEED}_"
    f"{int(TRAIN_PROP * 100)}_{int(round((1 - TRAIN_PROP - TEST_PROP) * 100))}"
)
pkl_path = SPLIT_DIR / f"{stem}.pkl"
json_path = SPLIT_DIR / f"{stem}.json"
readme_path = SPLIT_DIR / "README.md"

# Also store pairing ids for join-based analyses (embeddings / JEPA).
cpi = adata.obs["cell_pairing_index"].astype(str).to_numpy()
payload = {
    "dataset_name": DATASET_NAME,
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "reference_h5ad": str(ref_h5ad_path),
    "n_rows": int(adata.n_obs),
    "splitting_mode": "stratified",
    "split_obs": list(SPLIT_OBS),
    "train_prop": TRAIN_PROP,
    "test_prop": TEST_PROP,
    "seed": SPLIT_SEED,
    "note": (
        "Matches perturbgen.train stratified_split seed=42. "
        "Indices are row positions into paired src/tgt datasets and h5ads."
    ),
    "train_indices": train_indices,
    "val_indices": val_indices,
    "test_indices": test_indices,
    "train_cell_pairing_index": cpi[train_indices],
    "val_cell_pairing_index": cpi[val_indices],
    "test_cell_pairing_index": cpi[test_indices],
}

with pkl_path.open("wb") as f:
    pickle.dump(payload, f, protocol=pickle.HIGHEST_PROTOCOL)

summary = {
    k: payload[k]
    for k in [
        "dataset_name",
        "created_utc",
        "reference_h5ad",
        "n_rows",
        "splitting_mode",
        "split_obs",
        "train_prop",
        "test_prop",
        "seed",
        "note",
    ]
}
summary.update(
    {
        "n_train": int(train_indices.size),
        "n_val": int(val_indices.size),
        "n_test": int(test_indices.size),
        "pkl_path": str(pkl_path),
    }
)
json_path.write_text(json.dumps(summary, indent=2) + "\n")

readme_path.write_text(
    "\n".join(
        [
            "# Frozen train / val / test splits",
            "",
            f"- Primary artifact: `{pkl_path.name}`",
            f"- Summary: `{json_path.name}`",
            "",
            "Load in Python:",
            "",
            "```python",
            "import pickle",
            f"with open(r'{pkl_path}', 'rb') as f:",
            "    split = pickle.load(f)",
            "train_idx, val_idx, test_idx = (",
            "    split['train_indices'], split['val_indices'], split['test_indices']",
            ")",
            "```",
            "",
            "Recipe matches `perturbgen.train` stratified split (`seed=42`, 0.9 train / 0.1 val,",
            "`cell_type_harmonized`) used by Notebook 3 `masking_split` runs. No test split.",
            "",
        ]
    )
)

counts = pd.DataFrame(
    {
        "split": ["train", "val", "test"],
        "n": [train_indices.size, val_indices.size, test_indices.size],
        "fraction": [
            train_indices.size / adata.n_obs,
            val_indices.size / adata.n_obs,
            test_indices.size / adata.n_obs,
        ],
    }
)
print("Reference h5ad:", ref_h5ad_path)
print("Wrote:", pkl_path)
print("Wrote:", json_path)
print("Wrote:", readme_path)
display(counts)
